In [ ]:
# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements.  See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership.  The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License.  You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied.  See the License for the
# specific language governing permissions and limitations
# under the License.


## Setup

[OrcaRouter](https://www.orcarouter.ai) exposes an OpenAI-compatible API at `https://api.orcarouter.ai/v1`.

Install the dependencies and set your `ORCAROUTER_API_KEY`:


In [ ]:
!pip install "apache-burr[start]" openai

In [ ]:
import os
from typing import Tuple

import openai  # OrcaRouter is OpenAI-compatible

from burr.core import action, State, Application

ORCAROUTER_BASE_URL = os.getenv("ORCAROUTER_BASE_URL", "https://api.orcarouter.ai/v1")
ORCAROUTER_MODEL = os.getenv("ORCAROUTER_MODEL", "orcarouter/auto")

assert "ORCAROUTER_API_KEY" in os.environ, "set ORCAROUTER_API_KEY first"

def _orcarouter_client() -> openai.OpenAI:
    return openai.OpenAI(
        base_url=ORCAROUTER_BASE_URL,
        api_key=os.environ["ORCAROUTER_API_KEY"],
    )

## Define Actions

We define two actions:
1. `human_input` -- this is the first one, it accepts a prompt from the outside and adds it to the state
2. `ai_response` -- this sends the full chat history to OrcaRouter's `chat/completions` endpoint and stores the reply

Note we're only ever touching the `openai` client and pointing its `base_url` at OrcaRouter.

In [ ]:
@action(reads=[], writes=["prompt", "chat_history"])
def human_input(state: State, prompt: str) -> Tuple[dict, State]:
    """Pulls human input from the outside world and adds it to the chat history."""
    chat_item = {"content": prompt, "role": "user"}
    return {"prompt": prompt}, state.update(prompt=prompt).append(chat_history=chat_item)


@action(reads=["chat_history"], writes=["response", "chat_history"])
def ai_response(state: State) -> Tuple[dict, State]:
    """Queries OrcaRouter with the chat history."""
    client = _orcarouter_client()
    content = (
        client.chat.completions.create(
            model=ORCAROUTER_MODEL,
            messages=state["chat_history"],
        )
        .choices[0]
        .message.content
    )
    chat_item = {"content": content, "role": "assistant"}
    return {"response": content}, state.update(response=content).append(chat_history=chat_item)

# Create the app

We create our app by adding our actions, then adding transitions. The agent loops forever between `human_input` and `ai_response`, accumulating a `chat_history` in state.

In [ ]:
app = (
    ApplicationBuilder().with_actions(
        human_input=human_input,
        ai_response=ai_response
    ).with_transitions(
        ("human_input", "ai_response"),
        ("ai_response", "human_input"),
    ).with_state(chat_history=[]).with_entrypoint("human_input").build()
)

# Run the app

To run the app, we call the `.run` function, passing in a stopping condition. In this case, we want it to halt after `ai_response`. It returns the result, and the resulting state.

In [ ]:
final_action, result, state = app.run(
    halt_after=["ai_response"],
    inputs={"prompt": "What is Apache Burr?"},
)
print(state["response"])